# 01 — Initial baseline

Train the very first model on week 0 data and inspect its core metrics +
fairness slice. This is what `retrain.run_retrain(week=0)` does on a
fresh registry.

In [ ]:
%load_ext autoreload
%autoreload 2

from modelgate.config import settings
from modelgate.data import load_raw, split_into_weeks, prepare_window_for_training
from modelgate.features import engineer, split_x_y
from modelgate.models import build_xgb_pipeline
from modelgate.evaluate import core_metrics, fairness_metrics

In [ ]:
df = load_raw()
weeks = split_into_weeks(df)
print([len(w) for w in weeks])

In [ ]:
train, val = prepare_window_for_training(weeks, end_week=0)
train_eng = engineer(train); val_eng = engineer(val)
x_train, y_train = split_x_y(train_eng)
x_val, y_val = split_x_y(val_eng)
print(x_train.shape, x_val.shape)

In [ ]:
pipe = build_xgb_pipeline(x_train)
pipe.fit(x_train, y_train)
proba = pipe.predict_proba(x_val)[:, 1]
core = core_metrics(y_val, proba)
fair = fairness_metrics(val_eng, y_val, proba)
print('core:', core.as_dict())
print('fair:', fair.as_dict())